***Multi Agent Collaboration for Finance Analysis***

In [10]:
!pip install crewai==1.15.1 crewai_tools==1.15.1 langchain_community==0.4.2

In [11]:
# !pip install -U openai langchain langchain-openai

In [12]:
# !pip install --upgrade pip setuptools

In [13]:
# Warning Control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import ScrapeWebsiteTool, SerperDevTool
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

In [15]:
load_dotenv()

True

In [16]:
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
os.environ["OPENROUTER_API_KEY"] = openrouter_api_key
os.environ["OPENROUTER_API_BASE"] = "https://openrouter.ai/api/v1"
os.environ["OPENROUTER_MODEL_NAME"] = 'mistralai/mistral-7b-instruct'

openrouter_llm = ChatOpenAI(
    model=os.environ["OPENROUTER_MODEL_NAME"],
    openai_api_base=os.environ["OPENROUTER_API_BASE"],
    openai_api_key=os.environ["OPENROUTER_API_KEY"]
)

In [17]:
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

Creating agent now!

In [18]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze market data in real-time"
        "to identify trands and predict market movements.",
    backstory="Specializing in financial markets, this agent"
              "uses statical modelling and machine learning",
    verbose=True,
    allow_delegation=True,
    tools=[scrape_tool, search_tool]
)

trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop and test various trading strategies based on insights from the Data Analyst.",
    backstory="Equipped with a deep understanding of financial markets and quantitative analysis, this agent devises and refines trading strategies.",
    verbose=True,
    allow_delegation=True,
    tools=[scrape_tool, search_tool]
)


execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade execution strategies based on approved trading strategies.",
    backstory="This agent specializes in analyzing the timing, price, and logistical details of potential trades.",
    verbose=True,
    allow_delegation=True,
    tools=[scrape_tool, search_tool]
)


risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models and market dynamics, this agent scrutinizes the potential risks of proposed trades.",
    verbose=True,
    allow_delegation=True,
    tools=[scrape_tool, search_tool]
)

## Tasks

In [19]:
data_analysis_task = Task(
    description="Continuously monitor and analyze market data for selected assets.",
    expected_output="Meaningful insights and alerts about significant market movements.",
    agent=data_analyst_agent
)

strategy_development_task = Task(
    description="Develop and refine trading strategies based on the insights from the Data Analyst and user-defined risk tolerance.",
    expected_output="A set of potential trading strategies that align with the user's risk tolerance.",
    agent=trading_strategy_agent
)

execution_planning_task = Task(
    description="Analyze approved trading strategies to determine the best execution methods for individual trades.",
    expected_output="Detailed execution plans suggesting how and when to execute trades.",
    agent=execution_agent
)

risk_assessment_task = Task(
    description="Evaluate the risks associated with the proposed trading strategies and execution plans.",
    expected_output="A comprehensive risk analysis report detailing potential risks and mitigation recommendations.",
    agent=risk_management_agent
)

## Creatign the Crew

In [21]:

from crewai import Process, Crew, LLM

manager_llm = LLM(
    model="openrouter/nvidia/nemotron-3-ultra-550b-a55b:free",
    base_url=os.environ["OPENROUTER_API_BASE"],
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0.7
)

financialTradingCrew = Crew(
    agents=[data_analyst_agent, trading_strategy_agent, execution_agent, risk_management_agent],
    tasks=[data_analysis_task, strategy_development_task, execution_planning_task, risk_assessment_task],
    manager_llm=manager_llm,
    process=Process.hierarchical,
    verbose=True
)


## Running the CREW

In [ ]:
financial_trading_input={
    'stock_selection': 'AAPL',
    'initial_capital': '100000',
    'risk_tolerance': 'Medium',
    'trading_strategy_preference': 'Day Trading',
    'new_impact_consideration': True
}

# Jupyter runs its own underlying asyncio event loop, and CrewAI detects it, 
# which throws a RuntimeError when trying to execute synchronous code like crew.kickoff() inside it
result = await financialTradingCrew.kickoff_async(inputs=financial_trading_input)
from IPython.display import Markdown
Markdown(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e6b5af48-0270-4722-85a5-75b0e6988ead                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Continuously monitor and analyze market data for selected assets.                                        │
│  ID: 50728115-2677-4c07-a1e2-64baea515fa7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Continuously monitor and analyze market data for selected assets.                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I'd be happy to help you continuously monitor and analyze market data for selected assets. However, I need to  │
│  know which specific assets you'd like me to monitor.                                                           │
│                                                                                                                 │
│  Could you please specify:                                                                                      │
│  1. **Which assets** you want to track (e.g., specific stocks like AAPL, TSLA; crypto like BTC, ETH; forex      │
│  pairs; commodities; indices)                                                                                   │
│  2. **What timeframes** you're interested in (real-time, daily, weekly)                                         │
│  3. **What types of insights/alerts** are most valuable to you (price movements, volume spikes, technical       │
│  indicators, news sentiment, earnings, etc.)                                                                    │
│  4. **Any specific thresholds** for alerts (e.g., >5% price change, unusual volume, RSI extremes)               │
│                                                                                                                 │
│  Once you provide these details, I can delegate the monitoring and analysis to our Data Analyst for continuous  │
│  tracking and meaningful insights.                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: OpenAI API call failed: OPENAI_API_KEY is required                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

An unknown error occurred. Please check the details below.
Error details: OPENAI_API_KEY is required


[CrewAIEventsBus] Warning: Event pairing mismatch. 'tool_usage_finished' closed 'agent_execution_started' (expected
'tool_usage_started')

Tool delegate_work_to_coworker executed with result: Error executing task with agent 'trade advisor'. Error: OPENAI_API_KEY is required...


╭─────────────────────────────────────── ✅ Tool Execution Completed (#2) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Output: Error executing task with agent 'trade advisor'. Error: OPENAI_API_KEY is required                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯